# ！！用 Test_ML 集群运行该脚本！！

In [0]:
# 测试文件路径（根据个人目录修改）
file_path = "/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_UAT/FEIYI/AllField_Testcase_All_Market.xlsx"

In [0]:
# 环境配置
bootstrap_servers = "10.249.209.11:9092,10.249.209.13:9092,10.249.209.16:9092"

# 默认数据文件
json_file_path = "/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_UAT/AllField_Test_HKG.json"

# 默认生成的json备份目录
OUTPUT_DIR = "/dbfs/mnt/mdm-database/testcase/"

# 默认Topic映射（IsCopy为FALSE或找不到配置时使用）
topic_mapping_Talend = {
    "AUS": "ConsumerTopic",
    "HKG": "Consumer_HK",
    "IDN": "Consumer_ID",
    "JPN": "Consumer_JP",
    "KOR": "Consumer_KR",
    "MYS": "Consumer_MY",
    "NZL": "Consumer_NZ",
    "PHL": "Consumer_PH",
    "SGP": "Consumer_SG",
    "THA": "Consumer_TH",
    "TWN": "Consumer_TW",
    "VNM": "Consumer_VN"
}

topic_mapping_Databricks = {
    "AUS": "Consumer_AU_TS",
    "HKG": "Consumer_HK_TS",
    "IDN": "Consumer_ID_TS",
    "JPN": "Consumer_JP_TS",
    "KOR": "Consumer_KR_TS",
    "MYS": "Consumer_MY_TS",
    "NZL": "Consumer_NZ_TS",
    "PHL": "Consumer_PH_TS",
    "SGP": "Consumer_SG_TS",
    "THA": "Consumer_TH_TS",
    "TWN": "Consumer_TW_TS",
    "VNM": "Consumer_VN_TS"
}

In [0]:
# 1. 安装依赖（仅首次运行，安静模式不输出日志）
%pip install openpyxl pandas --quiet
# 2. 导入库并读取
import os
import json
import copy
import pandas as pd
import re
import uuid as uuid_module
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType
from collections import defaultdict

# 用户数据文件名
file_name = os.path.basename(file_path).replace(".xlsx", "")

# 读取Data和CopyConfig
df_data = pd.read_excel(io=file_path, sheet_name="Data", header=0, dtype='string')
df_config = pd.read_excel(io=file_path, sheet_name="CopyConfig", header=0, dtype='string')

# 兼容Excel列名前后空格（例如"MarketCode ")

df_data.columns = df_data.columns.str.strip()
df_config.columns = df_config.columns.str.strip()

if df_data.empty:
    raise ValueError("Data sheet 无数据，请检查文件！")
if df_config.empty:
    print("⚠️ CopyConfig sheet 无数据")

print(f"Data sheet 行数: {len(df_data)}")
print(f"CopyConfig sheet 行数: {len(df_config)}")

# 读取默认数据文件
json_raw_str = dbutils.fs.head(json_file_path)
template_dict = json.loads(json_raw_str)
print(template_dict)

# ===================== 1. 递归删除空字段的函数 =====================
def delete_nested_key(obj, keys):
    if len(keys) == 1:
        if keys[0] in obj:
            del obj[keys[0]]
    else:
        key = keys[0]
        if key in obj and isinstance(obj[key], dict):
            delete_nested_key(obj[key], keys[1:])
            if not obj[key]:
                del obj[key]

# ===================== 2. 校验JSON格式的函数 =====================
def validate_json_format(result):
    """
    仅校验：result能否被正确序列化为标准JSON
    返回：(is_valid: bool, error_msg: str)
    """
    try:
        json.dumps(result, ensure_ascii=False)
        return True, ""
    except Exception as e:
        return False, f"JSON格式错误：{str(e)}"

# ===================== 3. 获取ConsumerId的函数 =====================
def get_consumer_id(result):
    try:
        return result["Consumer"]["SourceSystem"]["ConsumerId"]
    except:
        return "未知ConsumerId"

# 清理空值
def clean(obj):
    if isinstance(obj, dict):
        return {k: clean(v) for k, v in obj.items() if v not in (None, "", {})}
    elif isinstance(obj, list):
        return [clean(i) for i in obj if i not in (None, "", {})]
    else:
        return obj

# ===================== 4. 填充UUID =====================
def fill_uuid(result):
    """如果Header.DocumentUUID或Consumer.@RecordUUID为空，则自动生成UUID"""
    try:
        if not result.get("Header", {}).get("DocumentUUID"):
            result.setdefault("Header", {})["DocumentUUID"] = str(uuid_module.uuid4())
    except:
        pass
    try:
        if not result.get("Consumer", {}).get("@RecordUUID"):
            result.setdefault("Consumer", {})["@RecordUUID"] = str(uuid_module.uuid4())
    except:
        pass
    return result

# ===================== 5. 保存JSON到DBFS =====================
def safe_filename(value):
    """将空值或非法字符处理为安全文件名"""
    if pd.isna(value) or str(value).strip() == "":
        return "UNKNOWN"
    # 替换Windows/Unix非法字符
    return re.sub(r'[\\/:*?"<>|]', "_", str(value).strip())

def save_json_to_dbfs(result, case_no, source_desc):
    """
    按 MarketCode_BrandCode_SourceSystemCode_ConsumerId_CaseNo_DocumentUUID.json 格式保存
    """
    try:
        os.makedirs(OUTPUT_DIR, exist_ok=True)

        market_code = safe_filename(result.get("Consumer", {}).get("SourceSystem", {}).get("MarketCode"))
        brand_code = safe_filename(result.get("Consumer", {}).get("SourceSystem", {}).get("BrandCode"))
        source_system_code = safe_filename(result.get("Consumer", {}).get("SourceSystem", {}).get("@Code"))
        consumer_id = safe_filename(result.get("Consumer", {}).get("SourceSystem", {}).get("ConsumerId"))
        case_no_safe = safe_filename(case_no)
        document_uuid = safe_filename(result.get("Header", {}).get("DocumentUUID"))

        filename = f"{market_code}_{brand_code}_{source_system_code}_{consumer_id}_{case_no_safe}_{document_uuid}.json"
        filepath = os.path.join(OUTPUT_DIR, filename)

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        print(f"💾 JSON已保存: {filepath} | 来源: {source_desc}")
        return filepath
    except Exception as e:
        print(f"⚠️ JSON保存失败: {str(e)} | 来源: {source_desc}")
        return None

# ===================== 6. 生成单条JSON =====================
def generate_single_json(row, template_dict):
    result = copy.deepcopy(template_dict)
    for col in row.index:
        if col == "Case" or col == "IsCopy":
            continue
        value = row[col]
        keys = col.split(".")

        if pd.isna(value):
            delete_nested_key(result, keys)
        else:
            current = result
            for key in keys[:-1]:
                current = current.setdefault(key, {})
            final_key = keys[-1]

            if col == 'Consumer.SourceSystem.BrandCode' or col == 'Consumer.SourceSystem.DivisionCode':
                current[final_key] = f'{int(value):02d}'
            elif col == 'Consumer.Recognition.PersistentKeyUpdateList.PersistentKeyUpdate' or col == 'Consumer.PersonalData.HobbyList.Hobby' or col == 'Consumer.PersonalData.CustomAttributeList.CustomAttribute' or col == 'Consumer.ContactInformation.EMediaList.EMedia' or col == 'Consumer.ContactInformation.PhoneList.Phone' or col == 'Consumer.ContactInformation.AddressList.Address' or col == 'Consumer.OptInList.OptIn' or col == 'Consumer.CrossBrandOptInList.CrossBrandOptIn' or col == 'Consumer.ProgramList.Program' or col == 'Consumer.AuxiliaryAttributeList.AuxiliaryAttribute' or col == 'Consumer.CustomAttributeList.CustomAttribute' or col == 'Consumer.TermsAndConditionList.TermsAndCondition' or col == 'Consumer.RemarkList.Remark' or col == 'Consumer.NoteList.Note' or col == 'Consumer.CustomerGroupList.CustomerGroup' or col == 'Consumer.PersonalData.SkinConcernsList.SkinConcerns' or col == 'Consumer.PersonalData.MakeUpConcernList.MakeUpConcerns' or col == 'Consumer.PersonalData.HairTypeList.HairType' or col == 'Consumer.PersonalData.HairConcernsList.HairConcerns':
                current[final_key] = json.loads(value)
            else:
                current[final_key] = str(value)

    return clean(result)

# ===================== 5. 应用CopyConfig覆盖 =====================
def apply_config(row, config):
    """
    将config中的配置覆盖到row上，并替换ConsumerId中的MarketCode
    返回新的Series
    """
    new_row = row.copy()

    # 原MarketCode
    original_market_code = row.get("Consumer.SourceSystem.MarketCode", "")
    if pd.isna(original_market_code) or str(original_market_code) == "":
        consumer_id = row.get("Consumer.SourceSystem.ConsumerId", "")
        consumer_id_str = str(consumer_id) if not pd.isna(consumer_id) else ""
        if len(consumer_id_str) >= 5:
            original_market_code = consumer_id_str[2:5]

    # 新MarketCode
    new_market_code = config.get("MarketCode", original_market_code)
    if pd.isna(new_market_code):
        new_market_code = original_market_code

    # 覆盖字段
    if not pd.isna(config.get("MarketCode")) and str(config.get("MarketCode")) != "":
        new_row["Consumer.SourceSystem.MarketCode"] = str(config.get("MarketCode"))
    if not pd.isna(config.get("AffiliateCode")) and str(config.get("AffiliateCode")) != "":
        new_row["Consumer.SourceSystem.AffiliateCode"] = str(config.get("AffiliateCode"))
    if not pd.isna(config.get("SourceSystemCode")) and str(config.get("SourceSystemCode")) != "":
        new_row["Consumer.SourceSystem.@Code"] = str(config.get("SourceSystemCode"))
    if not pd.isna(config.get("BrandCode")) and str(config.get("BrandCode")) != "":
        new_row["Consumer.SourceSystem.BrandCode"] = str(config.get("BrandCode"))
    if not pd.isna(config.get("DivisionCode")) and str(config.get("DivisionCode")) != "":
        new_row["Consumer.SourceSystem.DivisionCode"] = str(config.get("DivisionCode"))
    if not pd.isna(config.get("RegTouchPointCode")) and str(config.get("RegTouchPointCode")) != "":
        new_row["Consumer.PersonalData.RegTouchPointCode"] = str(config.get("RegTouchPointCode"))

    # ConsumerId替换：原MarketCode替换为新MarketCode（仅替换第一次出现）
    original_market_code_str = str(original_market_code)
    new_market_code_str = str(new_market_code)
    if new_market_code_str != original_market_code_str and original_market_code_str != "":
        consumer_id = new_row.get("Consumer.SourceSystem.ConsumerId", "")
        consumer_id_str = str(consumer_id) if not pd.isna(consumer_id) else ""
        if original_market_code_str in consumer_id_str:
            new_consumer_id = consumer_id_str.replace(original_market_code_str, new_market_code_str, 1)
            new_row["Consumer.SourceSystem.ConsumerId"] = new_consumer_id

    return new_row

# ===================== 6. 主处理函数 =====================
def process_all(df_data, df_config, template_dict):
    all_results = []

    # 预处理CopyConfig：仅保留IsActive为TRUE的有效配置
    active_configs = []
    for _, cfg in df_config.iterrows():
        is_active = str(cfg.get("IsActive", "")).strip().upper()
        if is_active == "TRUE":
            active_configs.append(cfg)

    for index, row in df_data.iterrows():
        is_copy = str(row.get("IsCopy", "FALSE")).strip().upper() == "TRUE"
        market_code = row.get("Consumer.SourceSystem.MarketCode", "")
        market_code = str(market_code).strip() if not pd.isna(market_code) else ""
        case_no = row.get("Case", "")

        if is_copy and active_configs:
            # IsCopy=TRUE 时，对所有有效CopyConfig逐条生成
            for cfg in active_configs:
                new_row = apply_config(row, cfg)
                result = generate_single_json(new_row, template_dict)

                config_market_code = new_row.get("Consumer.SourceSystem.MarketCode", "")
                config_market_code = str(config_market_code).strip() if not pd.isna(config_market_code) else ""
                talend_topic = str(cfg.get("TalendTargetTopic", "")).strip() if not pd.isna(cfg.get("TalendTargetTopic")) else ""
                databricks_topic = str(cfg.get("DatabricksTargetTopic", "")).strip() if not pd.isna(cfg.get("DatabricksTargetTopic")) else ""

                is_valid, error_msg = validate_json_format(result)
                if is_valid:
                    result = fill_uuid(result)
                    save_json_to_dbfs(result, case_no, source_desc=f"Data第{index+1}行-Copy配置")
                    all_results.append({
                        "result": result,
                        "talend_topic": talend_topic if talend_topic else topic_mapping_Talend.get(config_market_code, ""),
                        "databricks_topic": databricks_topic if databricks_topic else topic_mapping_Databricks.get(config_market_code, ""),
                        "source": f"Data第{index+1}行-Copy配置"
                    })
                    print(f"第{index+1}行Copy配置处理完成 ✅")
                else:
                    consumer_id = get_consumer_id(result)
                    print(f"第{index+1}行Copy配置校验失败 ❌ | ConsumerId: {consumer_id} | 错误原因: {error_msg}")
        else:
            # IsCopy为FALSE或CopyConfig无有效配置：直接发送原始数据
            result = generate_single_json(row, template_dict)
            is_valid, error_msg = validate_json_format(result)
            if is_valid:
                result = fill_uuid(result)
                save_json_to_dbfs(result, case_no, source_desc=f"Data第{index+1}行-原始数据")
                all_results.append({
                    "result": result,
                    "talend_topic": topic_mapping_Talend.get(market_code, ""),
                    "databricks_topic": topic_mapping_Databricks.get(market_code, ""),
                    "source": f"Data第{index+1}行-原始数据"
                })
                print(f"第{index+1}行原始数据处理完成 ✅")
            else:
                consumer_id = get_consumer_id(result)
                print(f"第{index+1}行原始数据校验失败 ❌ | ConsumerId: {consumer_id} | 错误原因: {error_msg}")

    print(f"\n处理完成！共生成 {len(all_results)} 条合法数据")
    return all_results

# 兼容本地/静态检查环境，确保spark已定义
if "spark" not in globals():
    spark = SparkSession.builder.getOrCreate()

# 执行处理
results = process_all(df_data, df_config, template_dict)

# 按Topic分组发送
topic_groups = defaultdict(list)
for item in results:
    if item["talend_topic"]:
        topic_groups[item["talend_topic"]].append(item["result"])
    if item["databricks_topic"]:
        topic_groups[item["databricks_topic"]].append(item["result"])

for topic, data_list in topic_groups.items():
    if not topic or not data_list:
        continue

    print(f"\n🚀 准备发送到Topic: {topic}，消息数: {len(data_list)}")

    try:
        json_strings = [json.dumps(item, ensure_ascii=False) for item in data_list]
        df_kafka = spark.createDataFrame(json_strings, StringType()).toDF("value")

        print(f"⏳ 正在发送到Topic: {topic}")
        df_kafka.write.format("kafka") \
            .option("kafka.bootstrap.servers", bootstrap_servers) \
            .option("kafka.request.timeout.ms", "15000") \
            .option("kafka.max.block.ms", "20000") \
            .option("kafka.delivery.timeout.ms", "30000") \
            .option("kafka.retries", "0") \
            .option("topic", topic) \
            .mode("append") \
            .save()

        print(f"✅ 已发送到：{topic}")
    except Exception as e:
        print(f"❌ 发送失败：{topic} | 错误: {str(e)}")

print("\n🎉 全部发送完成！")

In [0]:
'''
1. file_path
    测试数据文件.
    
    文件放置个人目录下:
    /Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_UAT/CHENLING/
    /Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_UAT/FEIYI/
    /Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_UAT/GETING/
    /Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_UAT/JINJIE/
    /Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_UAT/HUHUAN
    /Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_UAT/LVHENG

2. json_file_path
    json测试数据模板文件.
    公用同一文件,不需要修改.
    有特殊逻辑则修改后放置到上述个人目录下.

3. topic_mapping_Talend / topic_mapping_Databricks
    默认Topic映射，当IsCopy为FALSE或找不到CopyConfig配置时使用。
    按Consumer.SourceSystem.MarketCode匹配。

4. CopyConfig逻辑
    - IsActive为TRUE且非空时视为有效配置
    - 当Data行IsCopy=TRUE时，遍历所有有效CopyConfig逐条生成
    - 覆盖字段：MarketCode、AffiliateCode、SourceSystemCode(@Code)、BrandCode、DivisionCode、RegTouchPointCode
    - ConsumerId会自动替换其中第一次出现的原MarketCode子串
    - 发送Topic优先使用CopyConfig中指定的TalendTargetTopic/DatabricksTargetTopic，若为空则回退到默认映射
    - Header.DocumentUUID和Consumer.@RecordUUID为空时自动生成UUID
    - 每条合法JSON在发送Kafka前会保存到 /dbfs/mnt/mdm-database/testcase/
'''